# Week 03: From Procedural Code to Objects

A lookup system built first as functions acting on raw data, then rebuilt as classes and objects.


## The running problem

We need a small lookup system for character records: search, count, retrieve, delete, display.


Each record has three fields: first name, last name, role. The data arrives as a *multidimensional list*: one inner list per character.

Lookup tables like this are a simple but powerful idea: they let us organize data so it can be found quickly and reliably. This is the running example for the rest of the week (and next week too).


In [1]:
st_characters = [
    ["Jim", "Hopper", "Chief of Police"],
    ["Eleven", "", "Psychokinetic Overachiever"],
    ["Dustin", "Henderson", "Science Enthusiast"],
    ["Lucas", "Sinclair", "Strategist"],
    ["Max", "Mayfield", "Skateboarder"],
    ["Will", "Byers", "Missing Child"],
    ["Mike", "Wheeler", "Leader"],
    ["Steve", "Harrington", "Cool Guy"],
    ["Nancy", "Wheeler", "Aspiring Journalist"],
    ["Jonathan", "Byers", "Photographer"],
    ["Joyce", "Byers", "Determined Mother"],
    ["Murray", "Bauman", "Private Investigator"],
    ["Yuri", "Ismaylov", "Pilot"],
    ["Robin", "Buckley", "Ice Cream Shop Worker"],
    ["Erica", "Sinclair", "Younger Sister"],
    ["Billy", "Hargrove", "Annoying Lifeguard"],
    ["Eddie", "Munson", "Metalhead"],
    ["Henry", "Creel", "Cult Leader"],
    ["Vekna", "", "Mind Flayer"],
    ["Scott", "Clarke", "Teacher"],
    ["Leo", "Irakliotis", "Demogorgon"],
]


## Plan A: functions over a list of lists

Keep the data as `st_characters`. Write functions that take that list and do the work.


Six functionalities we want:

1. **contains_prefix(prefix)**: return `True` if any first name starts with `prefix`.
2. **indices_prefix(prefix)**: return the indices of records whose first name starts with `prefix`.
3. **count_first_name_endswith(suffix)**: count how many first names end with `suffix`.
4. **get_by_last_name(last_name)**: return all records with a matching last name.
5. **delete(first_name, last_name)**: delete one matching record.
6. **display()**: print the records nicely.

This is a perfectly valid way to start. Below are the first two, implemented.


In [2]:
# Named constants replace "magic" index numbers
FIRST_NAME_INDEX = 0
LAST_NAME_INDEX = 1
ROLE_INDEX = 2


def contains_prefix(records, prefix):
    """
    Return True if at least one record has a first name
    that starts with the given prefix. Otherwise, return False.
    """
    found = False

    for rec in records:
        first = rec[FIRST_NAME_INDEX]      # look up the first name by position
        if first.startswith(prefix):
            found = True

    return found


def indices_prefix(records, prefix):
    """
    Return a list of indices for all records whose first name
    starts with the given prefix.
    """
    matches = []

    for i in range(len(records)):
        first = records[i][FIRST_NAME_INDEX]
        if first.startswith(prefix):
            matches.append(i)              # remember the position, not the record

    return matches


In [3]:
print("Any first name starts with 'L'? ->", contains_prefix(st_characters, "L"))
print("Indices where first name starts with 'L' ->", indices_prefix(st_characters, "L"))


Any first name starts with 'L'? -> True
Indices where first name starts with 'L' -> [3, 20]


## This plan will not scale

Four problems show up as soon as the program grows.


### 1. Positional data is fragile
We must remember that `rec[0]` is first name, `rec[1]` is last name, `rec[2]` is role. A malformed record can crash or silently misbehave.

### 2. The representation leaks everywhere
Every function has to know the exact structure of a record. Change that structure even slightly, and every function using it needs an update.

### 3. Duplicate-record problem
A character with several roles ends up stored several times, once per role, which invites spelling and spacing inconsistencies.

### 4. Rules are hard to enforce
A rule like "records must be unique by (first_name, last_name)" has nothing to enforce it. Raw lists do not know about rules.

These are design smells: the data and the operations that belong together are scattered across the program.

OOP pitch: **keep the thing together**, bundle data and behavior.


## What is an object?

An object bundles **state** (data) and **behavior** (what it can do), built from a blueprint called a **class**.


A class is not a program by itself: it is the blueprint. Objects are the copies built from that blueprint, and a program can build as many as it needs.

Take a `Friend`: someone with a name, a phone number, a birthday. Those properties are captured by a class blueprint. In Java, a very simple version looks like this:

```java
class Friend {
    String firstName;
    String lastName;
    String phoneNumber;
    Date dob;
}
```

The data inside an object is collectively its **state**. State is only half the picture: the other half is **behavior**, how the object interacts with the rest of the program.


## Defining a class in Python

A class is defined with `class`. Its constructor, `__init__`, sets up each new object's attributes.


Deconstructing the constructor, starting from the most literal version:

```python
def __init__(this_object, first_name, last_name, phone, dob):
    this_object.attribute1 = first_name
    this_object.attribute2 = last_name
    this_object.attribute3 = phone
    this_object.attribute4 = dob
```

Read this as: initialize *this object* with these *attributes*.

`__init__` is one of several **special methods** Python uses to manage objects. They are often called **dunder** methods, short for **d**ouble **under**score, because they begin and end with `__`.

Naming conventions that make this readable:

- Use `self` instead of `this_object`, everyone reading Python code expects it.
- Use descriptive attribute names, e.g. `self.first_name`.
- The attribute name and the `__init__` argument name are often the same word (`self.phone = phone`), but they do not have to match (`self.phone = phn` still works). Whatever is on each side of `=` is a separate variable, even when the names look alike.

```python
class Friend:
    def __init__(self, first_name, last_name, phone, dob):
        self.first_name = first_name
        self.last_name = last_name
        self.phone = phone
        self.dob = dob
```


In [4]:
class Friend:
    """Represents one contact in an address book."""

    def __init__(self, first_name: str, last_name: str, phone: str, dob: str) -> None:
        # Each parameter becomes an attribute: this is the object's state
        self.first_name = first_name
        self.last_name = last_name
        self.phone = phone
        self.dob = dob

    def introduce(self) -> str:
        """Return a one-line self-introduction: this is the object's behavior."""
        return f"Hi, I'm {self.first_name} {self.last_name}. You can reach me at {self.phone}."


## Creating an instance

Calling the class like a function builds one object and runs `__init__` automatically.


In [5]:
friend1 = Friend("John", "Doe", "555-1234", "1990-01-01")
print(friend1.introduce())


Hi, I'm John Doe. You can reach me at 555-1234.


`friend1.first_name` reads an **attribute**: it accesses state. `friend1.introduce()` calls a **method**: it triggers behavior.

Every instance gets its own copy of the attributes set in `__init__`. A second `Friend` object would have its own `first_name`, completely independent of `friend1`'s. Later in the week we will meet a kind of variable that instances *share* instead of owning individually.


## Back to the lookup problem: model records as objects

Instead of `[first, last, role]`, each character becomes a `Character` object. The whole collection becomes a `Cast` object.


What are the underlying data of this object, and what is its purpose?

For the *Stranger Things* example, the underlying data is a record of characters and their roles. The purpose is to add, edit, remove, and search characters, and to describe them.

A very simple starting design:

```python
class Cast:

    def __init__(self):
        self.__underlying = list()
```

If `__underlying` stayed a list of plain lists, it would look like this:

```text
[
    ["Frodo", "Baggins", "Whining Ring Bearer"],
    ["Leo", "Irakliotis", "Faithful Sauron Servant"],
]
```

That is the same positional arrangement from Plan A: `record[0]` is a first name, `record[1]` a last name, and so on. It is so 1970s, and it carries every problem from the friction list above straight into the new class.

In a genuinely object-oriented design, `__underlying` should be a list of *objects*, not a list of lists. That means designing an object to model one movie or series character.


### A first, bare-bones `Character`

In [6]:
class Character:
    """A movie or series character (first pass: no identity rules yet)."""

    def __init__(self, first_name: str, last_name: str, role: str) -> None:
        self.first_name = first_name
        self.last_name = last_name
        self.role = role


## Adding a `Character` to the `Cast`

The underlying field of `Cast` is now meant to hold `Character` objects. How do we add one?


In [7]:
class Cast:
    """A collection of Character objects (first pass: duplicates allowed)."""

    def __init__(self) -> None:
        # The underlying data structure is a list that will be
        # populated with Character objects
        self.__underlying = []

    def add_character(self, first_name: str, last_name: str, role: str) -> None:
        """Add a new character to the cast."""
        # Create a new Character object based on the provided info
        new_character = Character(first_name, last_name, role)
        # The new character goes to the end of the underlying list
        self.__underlying.append(new_character)


## A new problem: duplicates

```python
best_story_ever = Cast()
best_story_ever.add_character("Frodo", "Baggins", "ring bearer")
best_story_ever.add_character("Frodo", "Baggins", "landlord")
best_story_ever.add_character("Frodo", "Baggins", "oppressing employer")
best_story_ever.add_character("Frodo", "Baggins", "Gollum's buddy")
```

Four calls, four separate records: the same person now exists four times in `__underlying`, once per role. We would rather keep **one unique record** per character and collect the roles on it. Real database systems solve this with *data normalization*; we will do something simpler here.


## Fix: one record per character, roles collected in a list

Goal: keep one unique record per character, identified by `(first_name, last_name)`.

If the same character is added again with a different role, we attach the new role to the existing record instead of creating a duplicate.


In [8]:
class Character:
    """A character with a unique (first_name, last_name) identity.

    Roles are stored as a list so one character can have multiple roles.
    """

    def __init__(self, first_name: str, last_name: str, role: str) -> None:
        self.first_name = first_name
        self.last_name = last_name
        self.roles = [role]

    def add_role(self, role: str) -> None:
        """Add a new role if it is not already present."""
        if role not in self.roles:
            self.roles.append(role)

    def key(self) -> tuple:
        """Return a tuple that uniquely identifies this character."""
        return (self.first_name, self.last_name)

    def __str__(self) -> str:
        """Nice string representation for printing."""
        full_name = f"{self.first_name} {self.last_name}".strip()
        roles_str = ", ".join(self.roles)
        return f"{full_name}: {roles_str}"


class Cast:
    """A collection of unique Character records.

    Uniqueness rule: (first_name, last_name) must be unique.
    If the character already exists, the new role is attached to it.
    """

    def __init__(self) -> None:
        self.__underlying = []  # list[Character]

    def _find_index(self, first_name: str, last_name: str) -> int:
        """Return the index of the matching character, or -1 if not found."""
        target_key = (first_name, last_name)
        for i in range(len(self.__underlying)):
            if self.__underlying[i].key() == target_key:
                return i
        return -1

    def add_character(self, first_name: str, last_name: str, role: str) -> None:
        """Add a character if new; otherwise attach the role to the existing character."""
        idx = self._find_index(first_name, last_name)
        if idx == -1:
            self.__underlying.append(Character(first_name, last_name, role))
        else:
            self.__underlying[idx].add_role(role)

    def describe(self) -> str:
        """Return a multi-line string describing the cast."""
        lines = [str(char) for char in self.__underlying]
        return "\n".join(lines)


In [9]:
best_story_ever = Cast()

best_story_ever.add_character("Frodo", "Baggins", "ring bearer")
best_story_ever.add_character("Frodo", "Baggins", "landlord")
best_story_ever.add_character("Frodo", "Baggins", "oppressing employer")
best_story_ever.add_character("Frodo", "Baggins", "Gollum's buddy")

print(best_story_ever.describe())


Frodo Baggins: ring bearer, landlord, oppressing employer, Gollum's buddy


## What makes a record unique? Identifiers

To manage data correctly, each record should have an **identifier**: a value, or group of values, that uniquely identifies it.

### Everyday examples
- **SSN**: identifies a person (mostly unique within a country).
- **UVID / Student ID**: identifies a student within a university.
- **VIN**: identifies a car.
- **ISBN**: identifies a book.
- **MAC address**: identifies a network device.

### Local vs. global uniqueness

**Local uniqueness** means unique within one system or organization. `JSmith@luc.edu` and `JSmith@uic.edu` can both exist; usernames, employee IDs, and student IDs are usually locally unique.

**Global uniqueness** means unique everywhere. MAC addresses, VINs, and ISBNs are designed so no duplicates exist worldwide.

### In our program

We use `(first_name, last_name)` as a **local identifier**: unique inside our program, though it may not be unique in real life (two real people can share a name). Designing good identifiers helps prevent duplicates, reduce errors, and keep data consistent.


## Time to draw the design: UML

We now have two main ideas:

- A **Character** represents one record: a unique name, many roles.
- A **Cast** manages many `Character` records and provides lookup operations.

Before adding more features, it helps to draw the design.

![UML class diagram](./figures/cast_uml.png)

`Cast` is responsible for managing the collection. `Character` is responsible for record-level behavior: things that make sense for one character.


### Reading a UML class diagram

**UML**, the Unified Modeling Language, is a standardized way to sketch software designs so people can talk about them quickly. Think of it as a map of the code: it does not replace the code, but it makes the structure easier to see, and to argue about.

A class diagram uses a box per class, divided into three parts:

1. Class name
2. Attributes (fields)
3. Methods (behaviors)

Notation used here:

- Visibility markers: `+` public, `-` private.
- Attribute format: `attribute_name: Type`.
- Method format: `method_name(param1: Type, param2: Type) : ReturnType`.
- Relationships between classes: coming in a later week.


## Public vs. private: the "all adults" principle

The UML diagram marks `__underlying` with `-`, private. That is **encapsulation**, one of the four classic pillars of OOP, alongside abstraction, inheritance, and polymorphism.

Encapsulation hides internal details so code outside the class does not depend on fragile internals, protects the rules the class needs to stay valid, and lets the implementation change later without breaking other code.

Our `Cast` has a rule to protect: one unique record per `(first_name, last_name)`. If outside code could reach in directly, `cast.__underlying.append(Character(...))`, it could bypass `add_character(...)` and create duplicates anyway. Keeping `__underlying` private forces every change through methods that can enforce the rule.

Python's privacy is a social contract rather than a lock. `_name` signals "internal use" by convention; `__name` triggers *name mangling*, which makes accidental access harder but not impossible. Unlike Java or C++, where private truly means inaccessible from outside, Python trusts the programmer. This is sometimes called the **"all adults" principle**: nothing stops you from reaching in anyway, the language just expects you not to.

The same convention applies to methods. `_find_index(...)` is a private helper `Cast` uses internally; outside code should not need it. Keeping it private-by-convention keeps the class's public interface small: a few methods that matter, with the implementation details out of the way.


## Special methods ("dunder" methods)

Python calls these automatically in specific situations: `__init__`, `__str__`, `__repr__`, `__eq__`, `__lt__`.


Each one has a trigger:

- `__init__(self, ...)`: runs when the class is called, `Character(...)`.
- `__str__(self)`: runs when the object meets `str(...)` or `print(...)`. It should read well for a person.
- `__repr__(self)`: runs when the object meets `repr(...)`, and whenever Python needs to show an object inside a container, such as printing a `list` of them. When there is nothing fancier to say, `__repr__` can simply delegate to `__str__`.
- `__eq__(self, other)`: runs for `==`. Without it, `==` falls back to comparing identity (are these the *same object in memory*), which is rarely what we mean for two records that merely look alike.
- `__lt__(self, other)`: runs for `<`, and is also what `sorted(...)` and `list.sort()` use to decide order. Defining `__lt__` alone is enough to make a list of custom objects sortable.

Special methods are what let a custom class behave like a Python built-in: printable, comparable, sortable, all with ordinary syntax instead of one-off function calls.


In [10]:
class Character:
    """A character with a unique (first_name, last_name) identity."""

    def __init__(self, first_name: str, last_name: str, role: str) -> None:
        self.first_name = first_name
        self.last_name = last_name
        self.roles = [role]

    def add_role(self, role: str) -> None:
        """Add a new role if it is not already present."""
        if role not in self.roles:
            self.roles.append(role)

    def key(self) -> tuple:
        """Return a tuple that uniquely identifies this character."""
        return (self.first_name, self.last_name)

    def __str__(self) -> str:
        full_name = f"{self.first_name} {self.last_name}".strip()
        roles_str = ", ".join(self.roles)
        return f"{full_name}: {roles_str}"

    def __repr__(self) -> str:
        # Delegate to __str__ so a list of Character objects prints nicely too
        return self.__str__()

    def __eq__(self, other: object) -> bool:
        """Two characters are equal if they share the same identity."""
        if not isinstance(other, Character):
            return NotImplemented
        return self.key() == other.key()

    def __lt__(self, other: "Character") -> bool:
        """Order characters by last name, then first name."""
        if not isinstance(other, Character):
            return NotImplemented
        return (self.last_name, self.first_name) < (other.last_name, other.first_name)


In [11]:
gandalf = Character("Gandalf", "the Grey", "wizard")
same_gandalf = Character("Gandalf", "the Grey", "guide")
bilbo = Character("Bilbo", "Baggins", "burglar")

print("gandalf == same_gandalf ->", gandalf == same_gandalf)
print("gandalf == bilbo        ->", gandalf == bilbo)

fellowship = [gandalf, bilbo]
print(sorted(fellowship))


gandalf == same_gandalf -> True
gandalf == bilbo        -> False
[Bilbo Baggins: burglar, Gandalf the Grey: wizard]


`gandalf == same_gandalf` is `True` even though they are two different objects in memory: `__eq__` compares `key()`, not identity. `sorted(fellowship)` works because Python calls `__lt__` behind the scenes, and printing the resulting list calls `__repr__` on each element, which is why `__repr__` delegating to `__str__` is convenient here.


## Class variables vs. instance variables

An **instance variable**, `self.first_name`, `self.roles`, belongs to one object. Every `Character` has its own.

A **class variable** is defined directly in the class body, not inside `__init__` tied to `self`, and is shared by every instance of the class. It is the right tool for something that belongs to the *class as a whole*: a running count of how many objects have been created, a default value every instance falls back to, a shared configuration setting.


### Class methods and static methods

A **class method**, marked with `@classmethod`, receives `cls` (the class itself) instead of `self`. It is the natural place to read or update a class variable, or to offer an alternate way of constructing an object.

A **static method**, marked with `@staticmethod`, receives neither `self` nor `cls`. It behaves like an ordinary function that happens to live inside the class for organization, callable as `Character.normalize_name(...)` without needing any instance at all.


In [12]:
class Character:
    """A character with a unique (first_name, last_name) identity."""

    DEFAULT_ROLE = "Extra"    # constant: used when no role is supplied
    _population = 0           # class variable: shared by every Character

    def __init__(self, first_name: str, last_name: str, role: str = None) -> None:
        self.first_name = first_name
        self.last_name = last_name
        self.roles = [role if role else Character.DEFAULT_ROLE]
        Character._population += 1   # one shared counter, updated by every instance

    def add_role(self, role: str) -> None:
        """Add a new role if it is not already present."""
        if role not in self.roles:
            self.roles.append(role)

    def key(self) -> tuple:
        """Return a tuple that uniquely identifies this character."""
        return (self.first_name, self.last_name)

    def __str__(self) -> str:
        full_name = f"{self.first_name} {self.last_name}".strip()
        roles_str = ", ".join(self.roles)
        return f"{full_name}: {roles_str}"

    def __repr__(self) -> str:
        return self.__str__()

    def __eq__(self, other: object) -> bool:
        if not isinstance(other, Character):
            return NotImplemented
        return self.key() == other.key()

    def __lt__(self, other: "Character") -> bool:
        if not isinstance(other, Character):
            return NotImplemented
        return (self.last_name, self.first_name) < (other.last_name, other.first_name)

    @classmethod
    def population(cls) -> int:
        """Return how many Character objects have been created so far."""
        return cls._population

    @staticmethod
    def normalize_name(name: str) -> str:
        """Return name with consistent capitalization, e.g. 'jim' -> 'Jim'."""
        return name.strip().title()


In [13]:
extra1 = Character(Character.normalize_name("  leo "), Character.normalize_name("irakliotis"), "demogorgon")
extra2 = Character("Scott", "Clarke")   # no role given: falls back to DEFAULT_ROLE

print(extra1)
print(extra2)
print("Characters created so far ->", Character.population())


Leo Irakliotis: demogorgon
Scott Clarke: Extra
Characters created so far -> 2


## Constants vs. magic values

A **magic value** is a literal dropped into code with no explanation: a bare `0`, a bare `"Extra"`. It works, but it forces a reader to guess what it means, and it invites the same value to be typed slightly wrong somewhere else.

A **named constant** fixes that: `FIRST_NAME_INDEX = 0` from Plan A, or `DEFAULT_ROLE = "Extra"` on `Character`, say what the value *means*, once, in one place. The convention in Python is `ALL_CAPS_WITH_UNDERSCORES`, which also signals "this is not meant to change while the program runs."

The rule of thumb: if a literal's meaning is not obvious from the line it appears on, give it a name.


## Recap

This week, in order:

- An object bundles state and behavior, built from a class blueprint.
- `__init__` sets up instance attributes; `self` refers to the object being built.
- Encapsulation, `__underlying`, keeps internals private by convention: the "all adults" principle.
- Special methods, `__str__`, `__repr__`, `__eq__`, `__lt__`, let a custom class print, compare, and sort like a built-in.
- Class variables are shared across every instance; class methods and static methods operate at the class level rather than on one object.
- Named constants replace magic values wherever a literal's meaning is not obvious on its own.

A bit more upfront design, choosing the right identifier, deciding what state belongs where, often leads to the right abstraction earlier. Good object-oriented design does not eliminate change, but it reduces how often we have to tear things apart and rebuild.


## Reading

- [Objects](https://learning.oreilly.com/library/view/introducing-python-3rd/9781098174392/ch11.html) from Bill Lubanovic's *Introducing Python,* 3rd edition.
- [Classes](https://docs.python.org/3/tutorial/classes.html) from the official Python tutorial.
- [Object-oriented design](https://learning.oreilly.com/library/view/python-object-oriented-programming/9781836642596/text/ch01.xhtml#chapter-1-object-oriented-design) from Lott and Phillips.
- **Must read:** [PEP 8, Readability counts](https://peps.python.org/pep-0008/), the official Python style guide. It offers real insight into the language, and it is worth reading closely if you plan to write Python (or supervise AI writing it for you) after this course.


## One more idea before we go: what is a data structure?

A `Cast` object is already a small data structure: it takes a plain list and imposes meaning on it (uniqueness, roles, lookup). Next week we give that idea a name, an **Abstract Data Type**, and build one properly: the Lookup Table.

For today, the general idea: a data structure reconciles how humans organize data with how computers store it. Computer memory is a linear arrangement of bytes, one piece of information after another, with no structure of its own. Any structure we want, we have to build.

Here are two views of the same data:

![](./figures/passengers.png)

On the left, the seating chart of a small airplane. On the right, the passenger manifest. The manifest alone does not say how passengers are seated, that is what the chart adds. Computer memory is more like the manifest: one piece of information after another. Structure, and meaning, is something we impose.


In [14]:
passenger_manifest = [
    "Thorin",
    "Balin",
    "Dwalin",
    "Fili",
    "Kili",
    "Dori",
    "Nori",
    "Ori",
    "Oin",
    "Gloin",
    "Bifur",
    "Bofur",
    "Bombur",
    "Gandalf",
    "Bilbo",
]

# A structured view of the passenger manifest: this is essentially a mini
# data structure that takes a linear list and imposes a 2D structure on it.

SEATS_PER_ROW = 4
FIRST_SEAT = 65  # ASCII 'A'
i = 0
while i < len(passenger_manifest):
    row = 1 + i // SEATS_PER_ROW
    col = chr(FIRST_SEAT + i % SEATS_PER_ROW)
    print(f" Seat {row}{col}: {passenger_manifest[i]:10s}", end="")
    if i % SEATS_PER_ROW == SEATS_PER_ROW - 1:
        print()
        print()
    i += 1


 Seat 1A: Thorin     Seat 1B: Balin      Seat 1C: Dwalin     Seat 1D: Fili      

 Seat 2A: Kili       Seat 2B: Dori       Seat 2C: Nori       Seat 2D: Ori       

 Seat 3A: Oin        Seat 3B: Gloin      Seat 3C: Bifur      Seat 3D: Bofur     

 Seat 4A: Bombur     Seat 4B: Gandalf    Seat 4C: Bilbo     